In [2]:
# Installations
!pip install roboflow
!pip install albumentations


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()


from roboflow import Roboflow
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("malletbottle2").project("sampled-yd-object-detection")
version = project.version(2)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [4]:
# Augmentations/Preprocssing based on first run/Claude advice
import albumentations as A

occlusion_augs = [
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(0.05, 0.25),   # fractions of image size
        hole_width_range=(0.05, 0.25),
        fill=0, p=0.5,  
    )
]


In [5]:
from ultralytics import YOLO

# Load model
model = YOLO("yolov8n.pt")

In [ ]:
# Train
results = model.train(
    data="Sampled-YD-Object-Detection-2/data.yaml", 
    cfg="custom_hyperparameters.yaml",
    device="mps",
    augmentations = occlusion_augs,
)

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.172 🚀 Python-3.12.7 torch-2.14.1 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'CoarseDropout', 'p': 0.5, 'fill': 0.0, 'fill_mask': None, 'hole_height_range': (0.05, 0.25), 'hole_width_range': (0.05, 0.25), 'num_holes_range': (1, 4)}}], auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=custom_hyperparameters.yaml, channels_last=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Sampled-YD-Object-Detection-2/data.yaml, degrees=180, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False,

scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)


       1/70      17.2G      2.176      4.349      2.748         43        960: 60% ━━━━━━━───── 15/25 25.0s/it 1:02:24<4:10


KeyboardInterrupt: 

In [10]:
# Validate model
model_recent = YOLO("runs/detect/train-4/weights/best.pt")
results = model_recent.val(data="Sampled-YD-Object-Detection-2/data.yaml", visualize=True, save_json=True, imgsz=640, device="mps")

print(results.confusion_matrix.matrix)
print(results.box.maps)

Ultralytics 8.4.172 🚀 Python-3.12.7 torch-2.14.1 MPS (Apple M4)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 282.8±68.8 MB/s, size: 66.6 KB)
val: Scanning /Users/wonheeryu/ai-ml-take-home/Sampled-YD-Object-Detection-2/valid/labels.cache... 199 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 199/199 16.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 1.3it/s 9.7s0.7s
                   all        199        257      0.954      0.875      0.934      0.613
                bottle        110        151      0.947      0.826       0.89      0.573
                mallet        106        106      0.961      0.925      0.977      0.652
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 11.3ms postprocess per image
Saving /Users/wonheeryu/ai-ml-take-home/runs/detect/val-5/predictions.json...

Evaluating faster-coco-eval mAP...
E

In [7]:
best_weights = "runs/detect/train-4/weights/best.pt"
results = YOLO(best_weights).predict(source="sample_images/", conf=0.45, save=True)
for r in results :
    r.show()


image 1/2 /Users/wonheeryu/ai-ml-take-home/sample_images/mallet.png: 640x640 1 mallet, 93.2ms
image 2/2 /Users/wonheeryu/ai-ml-take-home/sample_images/water.png: 640x448 1 bottle, 44.9ms
Speed: 2.1ms preprocess, 69.0ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 448)
Results saved to /Users/wonheeryu/ai-ml-take-home/runs/detect/predict-2


In [ ]:
# Get parameter count
import time
model.info()

start_time = time.time()
results = model("sample_images/")
end_time = time.time()


inference_speed = results[0].speed
print(f"Pure Inference TIme: {inference_speed['inference']:.2f} ms") # On macbook hardware

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs

image 1/2 /Users/wonheeryu/ai-ml-take-home/sample_images/mallet.png: 640x640 1 scissors, 1 toothbrush, 96.2ms
image 2/2 /Users/wonheeryu/ai-ml-take-home/sample_images/water.png: 640x448 1 bottle, 56.3ms
Speed: 5.8ms preprocess, 76.3ms inference, 8.4ms postprocess per image at shape (1, 3, 640, 448)
Pure Inference TIme: 96.19 ms
